# QuantJourney SDK - Global Macro Source Coverage

This notebook demonstrates a QuantJourney SDK workflow that queries US, global and regional macro connectors to build a source coverage matrix for macro research.

It covers:

- Direct QuantJourney SDK calls for the required market, macro, regulatory or portfolio data
- Transparent pandas/numpy calculations so research assumptions stay visible
- Chart-ready output that can be reused in notebooks, reports or API documentation

## Prerequisites

Make sure you have:

- Access to QuantJourney API (https://api.quantjourney.cloud)
- `QJ_API_KEY` configured in your environment
- Tenant access to the connectors used by this example

## Imports and Plot Style

In [ ]:
import os
import math
import json
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quantjourney.sdk import QuantJourneyAPI
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (12, 6), 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})


## QuantJourney Client

In [ ]:
qj = QuantJourneyAPI(api_key=os.environ['QJ_API_KEY'])
START = os.getenv('QJ_EXAMPLE_START', '2020-01-01')
END = os.getenv('QJ_EXAMPLE_END') or pd.Timestamp.today().normalize().strftime('%Y-%m-%d')


## Response Helpers

In [ ]:
def unwrap(payload: Any) -> Any:
    """Return the useful data value from common QuantJourney response shapes."""
    if isinstance(payload, dict) and 'data' in payload:
        payload = payload['data']
    if isinstance(payload, dict) and 'value' in payload:
        return payload['value']
    return payload

def as_rows(payload: Any) -> list[dict[str, Any]]:
    value = unwrap(payload)
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        for key in ('rows', 'data', 'items', 'prices', 'results'):
            if isinstance(value.get(key), list):
                return value[key]
        return [value]
    return []


In [ ]:
macro_calls = {'fred_cpi': qj.fred.get_cpi(), 'fred_10y': qj.fred.get_treasury_10y(), 'imf_gdp': qj.imf.get_gdp_data(country='US'), 'imf_inflation': qj.imf.get_inflation_data(country='US'), 'oecd_cpi': qj.oecd.get_cpi_data(country='USA'), 'worldbank_gdp': qj.worldbank.get_indicator(country='US', indicator='NY.GDP.MKTP.CD'), 'dbnomics_inflation': qj.dbnomics.get_inflation_rates(country='US'), 'dbnomics_rates': qj.dbnomics.get_interest_rates(country='US'), 'eurostat': qj.eurostat.get_eu_data()}


In [ ]:
coverage = []
for name, payload in macro_calls.items():
    rows = as_rows(payload)
    value = unwrap(payload)
    coverage.append({'source': name, 'available': payload is not None, 'rows': len(rows), 'shape': type(value).__name__})
coverage_df = pd.DataFrame(coverage).sort_values(['available', 'rows'], ascending=False)
display(coverage_df)


In [ ]:
coverage_df.set_index('source')['rows'].plot(kind='bar', title='Macro connector row coverage')
plt.ylabel('rows returned')
plt.show()


## Notes

This is an example workflow. In production, tenant scopes, connector allowlists,
provider metadata, request IDs and audit logs should be retained next to the resulting
tables or charts.